### Biblioteki

In [ ]:
import numpy as np
import pandas as pd
import plotly as plt
import dash
import json a

### Import

In [ ]:

with open(r'Dane/PC-AFK-K-03.01.2026-.json', 'r', encoding='utf-8') as f: #Scieżka
    raw_data = json.load(f)

bucket_name = "aw-watcher-afk_Pc-Gaming-RTX" #Nazwa bucketa w której jest wewnętrzna nazwa urządzenia
events_list = raw_data['buckets'][bucket_name]['events']

df1 = pd.json_normalize(events_list)
# df1['timestamp'] = pd.to_datetime(df1['timestamp'])

with open(r'Dane/PC-APP-K-03.01.2026-.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

bucket_name = "aw-watcher-window_Pc-Gaming-RTX"
events_list = raw_data['buckets'][bucket_name]['events']

df2 = pd.json_normalize(events_list)


### Ultimate APP + AFK

In [ ]:
df_afk = df1
df_window = df2

# 1. Konwersja na datetime (bardzo ważne!)
df_afk['timestamp'] = pd.to_datetime(df_afk['timestamp'],format='ISO8601')
df_window['timestamp'] = pd.to_datetime(df_window['timestamp'],format='ISO8601')

# 2. Obliczenie czasu ZAKOŃCZENIA każdego zdarzenia
# To kluczowe, żeby wiedzieć, gdzie zdarzenie się urywa
df_afk['end_time'] = df_afk['timestamp'] + pd.to_timedelta(df_afk['duration'], unit='s')
df_window['end_time'] = df_window['timestamp'] + pd.to_timedelta(df_window['duration'], unit='s')

# 1. Tworzymy listę WSZYSTKICH unikalnych punktów w czasie (starty i końce obu tabel)
all_timestamps = pd.concat([
    df_window['timestamp'], 
    df_window['end_time'], 
    df_afk['timestamp'], 
    df_afk['end_time']
]).unique()

df_master = pd.DataFrame({'timestamp': all_timestamps})
df_master = df_master.sort_values('timestamp').reset_index(drop=True)

# 2. Tworzymy nową "Główną Ramkę" na bazie tych punktów
df_master = pd.DataFrame({'timestamp': all_timestamps})

# 3. Używamy merge_asof (łączenie asynchroniczne)
# Szukamy "najnowszego zdarzenia, które zaczęło się przed danym punktem"

# Dołączamy informacje o oknie
df_window=df_window.rename(columns={
    'data.app' : 'app',
    'data.title' : 'title'
})

df_master = pd.merge_asof(
    df_master.sort_values("timestamp"), 
    df_window[['timestamp', 'end_time', 'app', 'title']].sort_values("timestamp"), # bierzemy tylko potrzebne kolumny
    on='timestamp', 
    direction='backward',
    suffixes=('', '_win') # suffix, żeby odróżnić end_time okna od end_time afk
)

# Dołączamy informacje o AFK

df_afk=df_afk.rename(columns={
    'data.status' : 'status',

})

df_master = pd.merge_asof(
    df_master, 
    df_afk[['timestamp', 'end_time', 'status']].sort_values("timestamp"), 
    on='timestamp', 
    direction='backward',
    suffixes=('', '_afk')
)

# 1. Filtrowanie "Duchów" (merge_asof mógł przeciągnąć zdarzenie poza jego czas trwania)
# Sprawdzamy, czy aktualny timestamp jest wciąż mniejszy niż czas zakończenia zdarzenia

# Czy okno jest wciąż aktywne?
is_window_active = df_master['timestamp'] < df_master['end_time']
# Czy status AFK jest wciąż aktywny?
is_afk_active = df_master['timestamp'] < df_master['end_time_afk']

# Zostawiamy tylko te momenty, gdzie FAKTYCZNIE było otwarte jakieś okno
df_final = df_master[is_window_active].copy()

# 2. Obsługa braku danych AFK
# Jeśli w danym momencie nie było żadnego wpisu w bucket AFK, uznajemy, że było 'not-afk'
# (lub zostawiamy NaN, zależnie od preferencji). Tutaj ustawiam domyślny status jeśli brak pokrycia.
df_final.loc[~is_afk_active, 'status'] = 'not-afk (unknown)'

# 3. Obliczenie czasu trwania nowych, pociętych kawałków
# Czas trwania to różnica między aktualnym wierszem a następnym
df_final['duration'] = df_final['timestamp'].diff().shift(-1).dt.total_seconds()

# Usuwamy ostatni wiersz (może być pusty/NaN) i wiersze z zerowym czasem
df_final = df_final[df_final['duration'] > 0]

# 4. Wybór finalnych kolumn
df = df_final[['timestamp', 'duration', 'app', 'title', 'status']]




In [ ]:
df

In [ ]:
pd.set_option('display.max_rows', 100)
df.loc[(df.loc[:,"status"]=="not-afk (unknown)"),:].loc[1:10000,:]

In [ ]:
wynik = df[df['status'] == 'not-afk'].groupby('app')['duration'].sum()
wynik.sort_values(ascending=False)

In [ ]:
print(df2.groupby("data.app")["duration"].sum().sort_values(ascending=False))
